In [1]:
import argparse
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import glob
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from matplotlib.colors import LogNorm, TwoSlopeNorm


sipm_channels = ([4,5,6,7,8,9] + \
                [10,11,12,13,14,15] + \
                [20,21,22,23,24,25] + \
                [26,27,28,29,30,31] + \
                [36,37,38,39,40,41] + \
                [42,43,44,45,46,47] + \
                [52,53,54,55,56,57] + \
                [58,59,60,61,62,63]) # Active channels

In [2]:
# Aggregate data for different voltages
def agg_data(file_names, voltages):

    # Check if the values provided for voltage match the file names (in number) # Might add check for correct values
    if len(file_names) != len(voltages):
        raise ValueError("File names and provided voltages do not match in length")
        return

    gains = np.zeros((len(file_names),8,64,3)) # First axis is bias voltages
    for idx, file_name in enumerate(file_names):

        data = np.load(file_name)
        gains[idx,...] = data['data']

    return gains

# Check chi squared value
def check_chisquared(array, sipm_channels):
    """ Provide a numpy array of shape (V,8,64,2) for (n_voltages, n_adc, n_ch, dim2) with in the second dimension (gain, chi squared)"""
    gains = array[...,0]
    chisquared = array[...,1]

    # Loop over active channels and count the fits that are between 0.5 and 1.5 for each channel
    for adc in range(0, gains.shape[1]):

        for ch in range(0, gains.shape[2]):
            if ch not in sipm_channels:
                pass
            else:
                #print(f"ADC {adc}, ch {ch}, {chisquared[:,adc,ch]}")
                
                if sum(0.5 <= x <= 1.5 for x in chisquared[:,adc,ch]) >= 4:
                    print(adc, ch)
                

    return

# Fitting function
def linear_fit(gains, voltages, pdf_filename, plot=False, verbose=False):
    """ WLS fit based on reduced chi squared values obtained from the multiple Gaussian model."""
    sipm_channels = ([4,5,6,7,8,9] + \
                     [10,11,12,13,14,15] + \
                     [20,21,22,23,24,25] + \
                     [26,27,28,29,30,31] + \
                     [36,37,38,39,40,41] + \
                     [42,43,44,45,46,47] + \
                     [52,53,54,55,56,57] + \
                     [58,59,60,61,62,63]) # Active channels
    
    voltages = np.array(voltages)
    fit_res = np.zeros((8,64,2)) # Store the coefficients of the linear fit per adc per channel here
    r_squared = np.zeros((8,64)) # Coefficient of determination
    pvalues = np.zeros((8,64,2)) # Check if coefficients are significantly different from zero
    bdv_value = np.zeros((8,64))
   
    plots_per_fig = 4
    no_fit = 0
    with PdfPages(pdf_filename) as pdf_output:
        for adc in range(0, 8):

            for ch in range(0,gains.shape[2]):
                if ch not in sipm_channels: 
                    pass # Skip inactive channels

                else: # Fit for each channel separately, because the nr of good fits varies per channel
                    gains_per_chan = gains[:,adc,ch,:] # 2D array with gain+stats for all voltages (voltages, gains/stats)
                    if np.sum(gains_per_chan) == 0:
                        if verbose:
                            print(f"No values available for ADC {adc} ch {ch}")

                    else:
                        cut_1 = (gains_per_chan[:,0] !=0) # Take out zero gains bc those are failed/bad fits
                        cut_2 = (gains_per_chan[:,1] < 1.5) & (gains_per_chan[:,1] > .5) # We only want to look at fitted gains that had a 'good' chi squared value
                        cut_combi =  cut_1# * cut_2

                        gains_per_chan_v1 = gains_per_chan[cut_combi] 
                        if (gains_per_chan[~cut_combi].size != 0) and verbose:
                            print(f"We don't take these values into account for ADC {adc} ch {ch} \n", gains_per_chan[~cut_combi], "\n At voltages ", voltages[~cut_combi])

                        voltages_v1 = voltages[cut_combi]

                        cut_3 = gains_per_chan_v1.size > 0  # Skip arrays that are empty after our first cut
                        if cut_3 == 1:

                            # Now check if for increasing voltages, the gain decreases. If that happens, something went wrong and we don't want to use that value in the fit.
                            gains_per_chan_v2 = np.array([gains_per_chan_v1[0]]) # Store the first gain value in a new array
                            voltages_v2 = np.array(voltages_v1[0])

                            for idx_v in range(1,len(voltages_v1)): # Start checking from the second value

                                if gains_per_chan_v1[idx_v,0] < gains_per_chan_v2[-1,0]: # Skip if the gain is lower than the last one that passed the cut
                                    if verbose:
                                        print(f"Probably {voltages_v1[idx_v]} is a bad fit for ADC {adc} ch {ch}")
                                        print(f"Gain for {voltages_v1[idx_v-1]}V: {gains_per_chan_v1[idx_v-1,0]}")
                                        print(f"Gain for {voltages_v1[idx_v]}V: {gains_per_chan_v1[idx_v,0]}")
                                else:
                                    gains_per_chan_v2 = np.vstack([gains_per_chan_v2,gains_per_chan_v1[idx_v,:]]) # Collect approved gain values
                                    voltages_v2 = np.append(voltages_v2, voltages_v1[idx_v])
                        
                            # Temporarily remove monotone inscrease requirement
                            gains_per_chan_v2 = gains_per_chan_v1
                            voltages_v2 = voltages_v1
                            cut_4 = (len(gains_per_chan_v2) > 1)  # Only attempt a fit for 3+ datapoints
                            if cut_4 == 1:

                                # Weighted least squares with chi2
                                volt_w_const = sm.add_constant(voltages_v2)
                                weights = 1/(gains_per_chan_v2[:,2]**2)
                                model_WLS = sm.WLS(gains_per_chan_v2[:,0], volt_w_const, weights=weights)
                                res = model_WLS.fit()


                                fit_res[adc, ch, :] = res.params

                                r_squared[adc, ch] = res.rsquared
                                pvalues[adc, ch,:] = res.pvalues

                            else:
                                if verbose:
                                    print(f"No fit for adc {adc} channel {ch}")
                
                    # For plotting
                    if plot:
                        
                                        
                        if ch % plots_per_fig == 0: # Creates a new figure for every 4 channels

                            fig, axs = plt.subplots(2,2, figsize=(10,10))
                            axs = axs.flatten()

                        ax = axs[ch % plots_per_fig]
                        if np.sum(fit_res[adc, ch, :]) != 0: # If the model results are zero, no fit was made
                            x_fit = np.linspace(39,49,100)
                            y_fit = res.params[1]*x_fit + res.params[0]
                            ax.set_title(f"ADC {adc} Channel {ch}, $V_{{BD}}$={round(-1*res.params[0]/res.params[1],2)}")
                            bdv_value[adc,ch] = round(-1*res.params[0]/res.params[1],2)
                            ax.plot(x_fit, y_fit,linestyle='--', label=f"fit of ADC {adc} ch {ch}, $R^2$={round(res.rsquared,3)}")
                            ax.errorbar(voltages_v2, gains_per_chan_v2[:,0], yerr=gains_per_chan_v2[:,2],color='black', fmt='x', capsize=3,label=f"ADC {adc} ch {ch}")
                            ax.set_xlabel("$V_{bias}$")
                            ax.set_ylabel("Gain")                                    
                            ax.legend()
                            if (adc == 0) or (adc == 1):
                                ax.set_ylim(0,1500)
                            else:
                                ax.set_ylim(0, 4000)
                            ax.set_xlim(40,49)
                        
                        else: 
                            ax.text(0.5, 0.5, f"No WLS fit possible for ADC {adc} ch {ch}", ha='center', va='center')
                            no_fit += 1
                        # Saves the figure per 4 channels
                        if ch % plots_per_fig == (plots_per_fig-1) or (ch == 63): 
                            fig.tight_layout()    
                            pdf_output.savefig(fig)
                            plt.close(fig)            
    np.savez("fit_res_Co60_v01_10dB_50tick.npz", bdv_value=bdv_value)
    print("Number of channels for which the linear fit failed is: ", no_fit)
    return fit_res, r_squared, pvalues

In [3]:
def maximize_rsquared(gains, voltages, pdf_filename, plot=False, verbose=False):
    """ Try the WLS fit for multiple sets of gains per channel, to see which model gives the best R squared value."""
    sipm_channels = ([4,5,6,7,8,9] + \
                     [10,11,12,13,14,15] + \
                     [20,21,22,23,24,25] + \
                     [26,27,28,29,30,31] + \
                     [36,37,38,39,40,41] + \
                     [42,43,44,45,46,47] + \
                     [52,53,54,55,56,57] + \
                     [58,59,60,61,62,63]) # Active channels
    
    voltages = np.array(voltages)
    fit_res = np.zeros((8,64,2)) # Store the coefficients of the linear fit per adc per channel here
    r_squared = np.zeros((8,64)) # Coefficient of determination
    pvalues = np.zeros((8,64,2)) # Check if coefficients are significantly different from zero
   
    plots_per_fig = 4
    no_fit = 0
    with PdfPages(pdf_filename) as pdf_output:
        for adc in range(0, 8):

            for ch in range(0,64):
                if ch not in sipm_channels: 
                    pass # Skip inactive channels

                else: # Fit for each channel separately, because the nr of good fits varies per channel
                    gains_per_chan = gains[:,adc,ch,:] # 2D array with gain+stats for all voltages (voltages, gains/chisquared/std)
                    if np.sum(gains_per_chan) == 0:
                        if verbose:
                            print(f"No values available for ADC {adc} ch {ch}")

                    else:
                        cut_1 = (gains_per_chan[:,0] !=0) # Take out zero gains bc those are failed/bad fits
                        cut_2 = (gains_per_chan[:,1] < 1.5) & (gains_per_chan[:,1] > .5) # We only want to look at fitted gains that had a 'good' chi squared value
                        cut_combi =  cut_1# * cut_2

                        gains_per_chan_v1 = gains_per_chan[cut_combi] 
                        if (gains_per_chan[~cut_combi].size != 0) and verbose:
                            print(f"We don't take these values into account for ADC {adc} ch {ch} \n", gains_per_chan[~cut_combi], "\n At voltages ", voltages[~cut_combi])

                        voltages_v1 = voltages[cut_combi]
                        cut_3 = gains_per_chan_v1.size > 0  # Skip arrays that are empty after our first cut
                        if cut_3 == 1:

                            # Now check if for increasing voltages, the gain decreases. If that happens, something went wrong and we don't want to use that value in the fit.
                            gains_per_chan_v2 = np.array([gains_per_chan_v1[0]]) # Store the first gain value in a new array
                            voltages_v2 = np.array(voltages_v1[0])

                            for idx_v in range(1,len(voltages_v1)): # Start checking from the second value

                                if gains_per_chan_v1[idx_v,0] < gains_per_chan_v2[-1,0]: # Skip if the gain is lower than the last one that passed the cut
                                    if verbose:
                                        print(f"Probably {voltages_v1[idx_v]} is a bad fit for ADC {adc} ch {ch}")
                                        print(f"Gain for {voltages_v1[idx_v-1]}V: {gains_per_chan_v1[idx_v-1,0]}")
                                        print(f"Gain for {voltages_v1[idx_v]}V: {gains_per_chan_v1[idx_v,0]}")
                                else:
                                    gains_per_chan_v2 = np.vstack([gains_per_chan_v2,gains_per_chan_v1[idx_v,:]]) # Collect approved gain values
                                    voltages_v2 = np.append(voltages_v2, voltages_v1[idx_v])
                        
                            # Temporarily remove monotone inscrease requirement
                            gains_per_chan_v2 = gains_per_chan_v1
                            voltages_v2 = voltages_v1

                            cut_4 = (len(gains_per_chan_v2) > 2)  # Only attempt a fit for 3+ datapoints
                            if cut_4 == 1:
                                
                                # Try the fit for multiple sets of data points, removing the point with the largest uncertainty in gain
                                fit_res_temp = np.zeros(2)
                                pvalues_temp = np.zeros(2)
                                r_squared_temp = 0
                                
                                # Create a plot to compare the linear fit with the different sets of data points
                                #fig2, axs2 = plt.subplots(2,2, figsize=(10,10))
                                #axs2 = axs2.flatten()

                                for nr_data_points in range(3, len(gains_per_chan_v2)+1):

                                    if nr_data_points == len(gains_per_chan_v2): # We don't do a partition when evaluating the set of all available data points
                                        gains_per_chan_v3 = gains_per_chan_v2
                                        voltages_v3 = voltages_v2
                                    else:
                                        # We make a partition of the <nr_data_points> amount of data points with the smallest std.
                                        # WARNING: now the gains are not ordered anymore
                                        indices = np.argpartition(gains_per_chan_v2[:,2], nr_data_points)[:nr_data_points]
                                        gains_per_chan_v3 = gains_per_chan_v2[indices]
                                        voltages_v3 = voltages_v2[indices]

                                    # Weighted least squares with chi2 for the selected set of gain values
                                    volt_w_const = sm.add_constant(voltages_v3)
                                    weights = 1/(gains_per_chan_v3[:,2]**2)
                                    model_WLS = sm.WLS(gains_per_chan_v3[:,0], volt_w_const, weights=weights)
                                    res = model_WLS.fit()

                                    # Now we select the model with the highest R squared value for each channel
                                    if res.rsquared > r_squared_temp:
                                        fit_res_temp = res.params
                                        pvalues_temp = res.pvalues
                                        r_squared_temp = res.rsquared

                                    # Check visually
                                #    x_fit = np.linspace(39,49,100)
                                #    y_fit = res.params[1]*x_fit + res.params[0]
                                #    axs2[nr_data_points-3].set_title(f"ADC {adc} Channel {ch}, $V_{{BD}}$={round(-1*res.params[0]/res.params[1],2)}, # data points: {nr_data_points}")
                                #    axs2[nr_data_points-3].plot(x_fit, y_fit,linestyle='--', label=f"fit of ADC {adc} ch {ch}, $R^2$={round(res.rsquared,3)}")
                                #    axs2[nr_data_points-3].errorbar(voltages_v3, gains_per_chan_v3[:,0], yerr=gains_per_chan_v3[:,2],color='black', fmt='x', capsize=3,label=f"ADC {adc} ch {ch}")
                                #    axs2[nr_data_points-3].set_xlabel("$V_{bias}$")
                                #    axs2[nr_data_points-3].set_ylabel("Gain")
                                #    axs2[nr_data_points-3].legend()  
                                #plt.show()
                                #plt.close(fig2)            

                                # Save the values of the model with the highest R squared per channel
                                fit_res[adc, ch, :] = fit_res_temp
                                pvalues[adc, ch,:] = pvalues_temp
                                r_squared[adc, ch] = r_squared_temp
                         
                            else:
                                if verbose:
                                    print(f"No fit for adc {adc} channel {ch}")
                
                    # For plotting
                    if plot:
                        
                                        
                        if ch % plots_per_fig == 0: # Creates a new figure for every 4 channels

                            fig, axs = plt.subplots(2,2, figsize=(10,10))
                            axs = axs.flatten()

                        ax = axs[ch % plots_per_fig]
                        if np.sum(fit_res[adc, ch, :]) != 0: # If the model results are zero, no fit was made
                            x_fit = np.linspace(39,49,100)
                            y_fit = fit_res[adc, ch, 1]*x_fit + fit_res[adc, ch, 0]
                            ax.set_title(f"ADC {adc} Channel {ch}, $V_{{BD}}$={round(-1*fit_res[adc, ch, 0]/fit_res[adc, ch, 1],2)}")
                            ax.plot(x_fit, y_fit,linestyle='--', label=f"fit of ADC {adc} ch {ch}, $R^2$={round(res.rsquared,3)}")
                            ax.errorbar(voltages_v2, gains_per_chan_v2[:,0], yerr=gains_per_chan_v2[:,2],color='black', fmt='x', capsize=3,label=f"ADC {adc} ch {ch}")
                            ax.set_xlabel("$V_{bias}$")
                            ax.set_ylabel("Gain")                                    
                            ax.legend()
                            if (adc == 0) or (adc == 1):
                                ax.set_ylim(0,1500)
                            else:
                                ax.set_ylim(0, 4000)
                            ax.set_xlim(40,49)
                        
                        else: 
                            ax.text(0.5, 0.5, f"No WLS fit possible for ADC {adc} ch {ch}", ha='center', va='center')
                            no_fit += 1
                        # Saves the figure per 4 channels
                        if ch % plots_per_fig == (plots_per_fig-1) or (ch == 63): 
                            fig.tight_layout()    
                            pdf_output.savefig(fig)
                            plt.close(fig)            
    #np.savez("fit_res_max_rsquared.npz", fit_res=fit_res, r_squared=r_squared, pvalues=pvalues)
    print("Number of channels for which the linear fit failed is: ", no_fit)
    return fit_res, r_squared, pvalues

In [4]:
# Load data from Oct 28th
file_list_1031 = sorted(glob.glob("/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/*50tick_v01_gainvalues.npz"))
file_name = "voltage_scan_Co60_v01_10dB_50tick"
print(file_list_1031)
voltages_1031 = [45, 46, 47, 48, 49]
gains_1031 = agg_data(file_list_1031[1:], voltages_1031[1:])
#fit_res_1031, _, _ = maximize_rsquared(gains_1031, voltages_1031, plot=True, verbose=False)

['/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_45V_Co66946_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_46V_Co66945_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_47V_Co66939_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_48V_Co66941_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_49V_Co66943_50tick_v01_gainvalues.npz']


In [5]:
# Load data
file_list = sorted(glob.glob("/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/*50tick_v01_gainvalues.npz"))   
print(file_list)

voltages = [45,46,47,48,49]
gains = agg_data(file_list[1:], voltages[1:])

# Check chi squared requirement 
#check_chisquared(gains, sipm_channels)

# Do the linear fit
file_name = 'voltage_scan_data_Co60_v01_10dB_50tick.pdf'
fit_res, _,_ = linear_fit(gains, voltages[1:], file_name, plot=True, verbose=False)

['/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_45V_Co66946_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_46V_Co66945_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_47V_Co66939_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_48V_Co66941_50tick_v01_gainvalues.npz', '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/2025_Calibration/Co60_Files/Run2_10dB_SPE_49V_Co66943_50tick_v01_gainvalues.npz']
Number of channels for which the linear fit failed is:  14


In [6]:
# Use this for quick printing of values 
print(gains.shape)
gain_per_voltage = gains[1,...,0]*4
text_gain = '\n'.join((map(str, np.round(gain_per_voltage.flatten(),2))))
print(text_gain)

(4, 8, 64, 3)
0.0
0.0
0.0
0.0
977.18
1007.6
955.83
0.0
824.25
787.27
299.25
894.36
758.16
1236.8
175.5
691.34
0.0
0.0
0.0
0.0
0.0
548.89
846.02
876.47
947.1
705.64
943.99
847.66
951.09
877.02
778.33
933.01
0.0
0.0
0.0
0.0
919.9
793.63
847.99
961.53
577.56
548.46
838.42
983.68
971.57
732.46
853.82
901.99
0.0
0.0
0.0
0.0
903.23
924.3
578.79
914.78
857.31
946.42
902.36
930.27
992.87
914.79
1076.5
821.85
0.0
0.0
0.0
0.0
950.18
947.17
1029.79
977.17
928.42
957.39
908.31
1091.9
974.6
655.23
941.72
946.23
0.0
0.0
0.0
0.0
894.05
944.99
911.13
790.23
887.0
869.04
922.25
0.0
840.0
963.09
714.35
0.0
0.0
0.0
0.0
0.0
679.35
791.35
635.8
663.95
790.3
618.45
521.8
950.3
649.6
729.4
802.55
747.25
0.0
0.0
0.0
0.0
1018.24
720.68
587.3
753.2
1046.87
827.4
816.55
751.8
886.9
675.5
601.15
652.4
0.0
0.0
0.0
0.0
2480.18
2560.04
2366.73
2481.06
2593.51
2540.96
2544.94
2642.19
2824.19
2547.57
2662.89
2472.37
0.0
0.0
0.0
0.0
2568.29
2567.71
2475.64
2556.82
2604.79
2589.68
2273.77
2350.58
2502.11
2503.37
2382.86

In [7]:
# Use the version of the script that maximizes the WLS R squared.
values = maximize_rsquared(gains, voltages,  "0411/voltage_scan_max_R2_0411_10dB.pdf", plot=True, verbose=False)
rsq_fit = values[0]
rsq_fit[rsq_fit==0] = 200
rsq_fit = -1*rsq_fit[...,0]/rsq_fit[...,1]
text_rsq = '\n'.join((map(str, np.round(rsq_fit.flatten(),2))))
print(text_rsq)

/tmp/ipykernel_2253513/1539008907.py:19: MatplotlibDeprecationWarning: Keeping empty pdf files is deprecated since 3.8 and support will be removed two minor releases later.
  with PdfPages(pdf_filename) as pdf_output:


FileNotFoundError: [Errno 2] No such file or directory: '0411/voltage_scan_max_R2_0411_10dB.pdf'

In [ ]:
# BD Voltages
fit_res[fit_res==0] = 200
V = -1*fit_res[...,0]/fit_res[...,1]
#V = V[:,sipm_channels]
label = np.arange(0,8)
fig = plt.gcf()
fig.patch.set_facecolor('white')

plt.figure(figsize=(12,8))
plt.hist(V.T, bins=60, stacked=True, label=label)
plt.xlabel("$V_{breakdown}$")
plt.ylabel("Counts")
plt.legend(title='ADC',loc='upper center',ncol=2)
plt.show()
text = '\n'.join((map(str, np.round(V.flatten(),2))))
print(text)

In [ ]:
from matplotlib.patches import Rectangle
# Mask inactivate channels
mask = np.zeros(gains.shape[2])
mask[sipm_channels] = True

# Plot a heatmap per ADC
fig, axs = plt.subplots(nrows=4, ncols=2, figsize=(15,15))
axs = axs.flatten()
for adc in range(gains.shape[1]):
    gains_per_adc = gains[:,adc,:,0]
    ax = axs[adc]
    sns.heatmap(gains_per_adc.T, ax=ax, cmap='viridis',xticklabels=voltages[:gains.shape[0]], cbar_kws={'label': 'Gain'})
    for ch in range(gains.shape[2]):
        if mask[ch] == False:
            ax.add_patch(Rectangle((0,ch),width=gains.shape[0], height=1,color='dimgrey'))
    ax.set_title("ADC {}".format(adc))

fig.tight_layout()
plt.show()



fig, axs = plt.subplots(nrows=4, ncols=2, figsize=(15,12))
axs = axs.flatten()
vmin = 1e-6
vmax = 10
norm = TwoSlopeNorm(vmin=vmin, vcenter=1,vmax=vmax)
for adc in range(gains.shape[1]):
    stats_per_adc = gains[:,adc,:,1]#+1e-6 # To avoid log(0) in the log scale

    ax = axs[adc]
    sns.heatmap(stats_per_adc.T, ax=ax, cmap='viridis_r', norm=norm,xticklabels=voltages[:gains.shape[0]],cbar_kws={'label': r'$\chi^2$'})
    
    for ch in range(gains.shape[2]):
        if mask[ch] == False:
            ax.add_patch(Rectangle((0,ch),width=gains.shape[0],edgecolor='black', lw=1, height=1,color='dimgrey'))
    ax.set_title("Chi squared ADC {}".format(adc))

fig.tight_layout()
plt.show()

print(np.sum(gains[:,:,:,1]==1))

In [ ]:
# Heatmap for all sipms together

heatmap_data = np.transpose(gains_v2[...,0], (1,2,0)).reshape(8*64,gains_v2.shape[0])
sns.heatmap(heatmap_data, cmap='viridis', cbar=True, xticklabels=voltages_v2[:gains_v2.shape[0]])
plt.xlabel('$V_{bias}$')
plt.ylabel("Channel")
labels = np.arange(0,64)*8
plt.yticks(labels)
plt.show()
print(labels)

In [ ]:
# James's plotting code
def plot_metric_div(
    metric,
    channel_status=None,
    log_scale=False,
    exclude_bad=True,
    mask_inactive=True,
    title='Metric',
    center=None,
    cmap='bwr'
):

    # Set up figure
    fig, ax = plt.subplots(figsize=(20, 4))
    # Mask bad channels if requested
    if channel_status is not None and exclude_bad:
        metric = np.ma.masked_where(channel_status != 0, metric)
    # If not excluding, make inactive (-1) NaN so it uses "bad" color; keep bad>0 for overlay
    if channel_status is not None and mask_inactive:
        inactive_idx = np.where(channel_status == -1)
        metric[inactive_idx] = np.nan
    # Colormap / norm
    if cmap == 'rwb':
        # reverse blue-white-red
        cmap = plt.cm.get_cmap('bwr').copy()
        cmap = cmap.reversed()
    cmap = plt.cm.get_cmap(cmap).copy()
    cmap.set_bad('black')  # masked/NaN shown as black for divergent
    # centre of divergence (white)
    if center is None:
        print ("Center not specified, using 0.0")
        center = 0.0
    # Determine a symmetric span about `center` that covers data *and* (if set) the window edges
    data_min = np.nanmin(metric)
    data_max = np.nanmax(metric)
    candidates = [max(0.0, center - data_min), max(0.0, data_max - center)]
    half_span = max(candidates) if candidates else 1.0
    if not np.isfinite(half_span) or half_span <= 0:
        half_span = 1.0
    # log scale with SymLogNorm
    if log_scale:
        # Shift so that white (0 after shift) is at 'center'
        shifted = metric - center
        # Choose a linear threshold for SymLog (smallest nonzero |shifted| with a floor)
        nz = np.abs(shifted)
        nz = nz[nz > 0]
        linthresh = max(np.nanmin(nz) if nz.size else 0.0, half_span * 1e-3)
        norm = colors.SymLogNorm(
            linthresh=linthresh,
            vmin=-half_span,
            vmax=+half_span,
            base=10.0
        )

        im = ax.imshow(shifted, cmap=cmap, norm=norm, aspect='auto')

        # Colorbar shows original (unshifted) values
        cbar = plt.colorbar(im, ax=ax, label=title)
        cbar.formatter = ticker.FuncFormatter(lambda val, pos: f"{val + center:g}")
        cbar.update_ticks()
    else:
        # Linear divergent symmetric scaling
        norm = colors.TwoSlopeNorm(vmin=center - half_span, vcenter=center, vmax=center + half_span)
        im = ax.imshow(metric, cmap=cmap, norm=norm, aspect='auto')
        plt.colorbar(im, ax=ax, label=title)

    # Overlay bad channels if not excluded
    if channel_status is not None and not exclude_bad:
        color = 'k'
        bad_idx = np.where(channel_status > 0)
        for y, x in zip(*bad_idx):
            ax.scatter(x, y, s=100, c=color, marker='o', edgecolors='none', zorder=10)

    # Axes styling
    ax.set_xticks(np.arange(0, 64, 1) - 0.5)
    ax.set_yticks(np.arange(0, 8, 1) - 0.5)
    ax.set_xticklabels(np.arange(0, 64, 1))
    ax.set_yticklabels(np.arange(0, 8, 1))
    plt.xlabel('Channel')
    plt.ylabel('ADC')
    plt.grid(which='both', linewidth=0.5, alpha=0.5, color='white')
    plt.tight_layout()
    plt.title(title)
    plt.show()

In [ ]:
# Now fit

# First only for ADC7
data_adc7 = gains[:,7,...]
print(data_adc7[...,0].shape)
for chan in range(data_adc7.shape[1]):
    plt.scatter(voltages, data_adc7[:,chan,0])
plt.show()

# I want to include only non-zero values in my fit.
# Here I filter out zeros for each row(=voltage)

cleaned = []
for row in data_adc7[...,0]:
    values = row[row!=0]
    channels = np.arange(48)[row!=0] # Keep track of the channels
    cleaned.append((values, channels))

for idx, (values, channels) in enumerate(cleaned):
    plt.scatter(channels, values, label=voltages[idx])
plt.legend()
plt.show()

In [ ]:
# Actual fitting
fit_res_adc7 = []

plt.figure()

for ch in range(0,data_adc7.shape[1]):
    gains_per_chan = data_adc7[:,ch,0]
    mask = (gains_per_chan !=0)
    gains_per_chan_cleaned = gains_per_chan[mask]
    voltages_cleaned = voltages[mask]
    
    cut_1 = (len(gains_per_chan_cleaned) > 2)  # Let's only attempt a fit for 3+ datapoints
    indices_cleaned = np.where(mask)[0]
    cut_2 = (np.any(np.diff(indices_cleaned)) == 1) # Checks if the non-zero array has data for two consecutive voltages

    try:
        cut_3 = ((gains_per_chan_cleaned[-1] - gains_per_chan_cleaned[0]) > 0) # If the final value is smaller than the first, something must have gone wrong
    except:
        print("No fit for ch", ch)
        pass
    if cut_1 * cut_2 * cut_3 == 1:

        coeffs = np.polyfit(voltages_cleaned, gains_per_chan_cleaned, deg=1)
        fit_res_adc7.append(coeffs)
        f_lin = np.poly1d(coeffs)
        x_fit = np.linspace(min(voltages_cleaned), max(voltages_cleaned), 100)
        y_fit = f_lin(x_fit)
    #    plt.scatter(voltages_cleaned, gains_per_chan_cleaned)
        plt.plot(x_fit, y_fit, label=ch)
        
    else:
        print("No fit for channel {}".format(ch))
del coeffs
plt.legend()
plt.show()
fit_res = np.array(fit_res)
V_BD_adc7 = -fit_res_adc7[:,1]/fit_res_adc7[:,0]

In [ ]:
# Now for all ADC's
fit_res = np.zeros((8,48,2))

for adc in range(0, gains.shape[1]):

    for ch in range(0,data_adc7.shape[1]):

        gains_per_chan = gains[:,adc,ch,0]
        mask = (gains_per_chan !=0)
        gains_per_chan_cleaned = gains_per_chan[mask] # Take out zeros (per channel)
        voltages_cleaned = voltages[mask]
    
        cut_1 = (len(gains_per_chan_cleaned) > 2)  # Let's only attempt a fit for 3+ datapoints
        indices_cleaned = np.where(mask)[0]
        cut_2 = (np.any(np.diff(indices_cleaned)) == 1) # Checks if the non-zero array has data for two consecutive voltages

        try:
            cut_3 = ((gains_per_chan_cleaned[-1] - gains_per_chan_cleaned[0]) > 0) # If the final value is smaller than the first, something must have gone wrong
        except:
            print(f"No fit for ADC {adc} ch {ch}")
            pass
        if cut_1 * cut_2 * cut_3 == 1:

            coeffs = np.polyfit(voltages_cleaned, gains_per_chan_cleaned, deg=1)
            fit_res[adc, ch, :] = coeffs

            # For plotting
            if adc == 0: # I want to see what happens at ADC0 rn

                f_lin = np.poly1d(coeffs)
                x_fit = np.linspace(min(voltages_cleaned), max(voltages_cleaned), 100) 
                y_fit = f_lin(x_fit) 
                plt.plot(x_fit, y_fit) # Horrible

        else:
            print(f"No fit for adc {adc} channel {ch}")

In [ ]:
with np.errstate(divide='ignore', invalid='ignore'): # Because there WILL be zeros...
    V_BD = -fit_res[...,1]/fit_res[...,0]

In [ ]:
# How many presumably correct values do we have?
values_1 = V_BD[~np.isnan(V_BD)]
values_2 = values_1[(values_1>40) & (values_1<45)] # From the datasheet I expect the values to be at least in this range.

#print(values_2)
print(f"Values for {len(values_2)} out of 384" )
print("Mean V_BD is ", np.mean(values_2))

In [ ]:
print((np.sum(gains[:,:,:,1]) > 0.5 & np.sum(gains[;,:,:,1] < 1.5))

In [ ]:
teller = 0
for adc in range(0,8):
    for ch in range(0,48):
        #print(gains[:,adc,ch,1])
        for file in range(0,gains.shape[0]):
            if gains[file,adc,ch,1] > 0.5:
                if gains[file,adc,ch,1] < 1.5:
                    teller += 1
                    print(gains[file,adc,ch,1])

In [ ]:
teller

In [ ]:
import numpy as np
waveform_data = []
for files in range(0,3):
    waveform_data_temp = np.zeros((6000,8,64,200))
    waveform_data.append(waveform_data_temp)
waveform_data = np.concatenate(waveform_data)
waveform_data.shape

In [ ]:
x = np.concatenate(waveform_data)

In [ ]:
inputs = ['sfsd',
          'fsdfs']
for i in inputs:
    print(i)